# Task 1 - Preprocess and Explore the Data
**Objective:** Load, clean, and understand the financial data to prepare it for modeling.

## 1. Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.float_format', '{:.4f}'.format)

## 2. Extract Historical Financial Data

In [ ]:
TICKERS = ['TSLA', 'BND', 'SPY']
START_DATE = '2015-01-01'
END_DATE = '2026-06-30'

raw_data = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=False)
print(f"Downloaded data shape: {raw_data.shape}")
print(f"Date range: {raw_data.index.min()} to {raw_data.index.max()}")
raw_data.head()

In [ ]:
# Separate DataFrames per asset
dfs = {}
for ticker in TICKERS:
    df = raw_data.xs(ticker, axis=1, level=1).copy()
    df.index = pd.to_datetime(df.index)
    dfs[ticker] = df
    print(f"{ticker}: {df.shape}")

## 3. Data Cleaning and Understanding

In [ ]:
# Basic statistics
for ticker in TICKERS:
    print(f"\n{'='*50}")
    print(f"  {ticker} - Basic Statistics")
    print('='*50)
    print(dfs[ticker].describe())

In [ ]:
# Check data types and missing values
for ticker in TICKERS:
    print(f"\n--- {ticker} ---")
    print("Data types:")
    print(dfs[ticker].dtypes)
    print("\nMissing values:")
    print(dfs[ticker].isnull().sum())

In [ ]:
# Handle missing values using forward fill then backward fill
for ticker in TICKERS:
    before = dfs[ticker].isnull().sum().sum()
    dfs[ticker] = dfs[ticker].ffill().bfill()
    after = dfs[ticker].isnull().sum().sum()
    print(f"{ticker}: {before} missing values -> {after} after fill")

In [ ]:
# Ensure correct data types
for ticker in TICKERS:
    for col in ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']:
        if col == 'Volume':
            dfs[ticker][col] = dfs[ticker][col].astype(np.int64)
        else:
            dfs[ticker][col] = dfs[ticker][col].astype(np.float64)

print("Data types corrected.")
print(dfs['TSLA'].dtypes)

In [ ]:
# Save cleaned data
for ticker in TICKERS:
    dfs[ticker].to_csv(f'../data/processed/{ticker}_cleaned.csv')
print("Cleaned data saved to data/processed/")

## 4. Exploratory Data Analysis (EDA)

### 4.1 Closing Price Over Time

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
colors = {'TSLA': '#E31937', 'BND': '#1F77B4', 'SPY': '#2CA02C'}

for ax, ticker in zip(axes, TICKERS):
    ax.plot(dfs[ticker].index, dfs[ticker]['Adj Close'], color=colors[ticker], linewidth=1.2)
    ax.set_title(f'{ticker} - Adjusted Closing Price', fontsize=13, fontweight='bold')
    ax.set_ylabel('Price (USD)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

axes[-1].set_xlabel('Date')
plt.suptitle('Adjusted Closing Prices (2015 - 2026)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/closing_prices.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Daily Percentage Change (Volatility)

In [ ]:
for ticker in TICKERS:
    dfs[ticker]['Daily_Return'] = dfs[ticker]['Adj Close'].pct_change() * 100

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, ticker in zip(axes, TICKERS):
    returns = dfs[ticker]['Daily_Return'].dropna()
    ax.plot(returns.index, returns, color=colors[ticker], linewidth=0.7, alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'{ticker} - Daily % Return  |  Std: {returns.std():.2f}%', fontsize=12)
    ax.set_ylabel('Return (%)')

axes[-1].set_xlabel('Date')
plt.suptitle('Daily Percentage Returns (2015 - 2026)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/daily_returns.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Rolling Mean and Standard Deviation (30-day)

In [ ]:
WINDOW = 30

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for ax, ticker in zip(axes, TICKERS):
    price = dfs[ticker]['Adj Close']
    rolling_mean = price.rolling(WINDOW).mean()
    rolling_std = price.rolling(WINDOW).std()

    ax.plot(price.index, price, color=colors[ticker], linewidth=0.8, alpha=0.5, label='Adj Close')
    ax.plot(rolling_mean.index, rolling_mean, color='black', linewidth=1.5, label=f'{WINDOW}-day Mean')
    ax.fill_between(price.index,
                    rolling_mean - rolling_std,
                    rolling_mean + rolling_std,
                    alpha=0.2, color=colors[ticker], label='±1 Std Dev')
    ax.set_title(f'{ticker} - Rolling Mean & Volatility Band', fontsize=12)
    ax.set_ylabel('Price (USD)')
    ax.legend(fontsize=9)

axes[-1].set_xlabel('Date')
plt.suptitle(f'{WINDOW}-Day Rolling Statistics', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/rolling_stats.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 Return Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, ticker in zip(axes, TICKERS):
    returns = dfs[ticker]['Daily_Return'].dropna()
    ax.hist(returns, bins=80, color=colors[ticker], edgecolor='white', alpha=0.85)
    ax.axvline(returns.mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean: {returns.mean():.2f}%')
    ax.set_title(f'{ticker} Return Distribution', fontsize=12)
    ax.set_xlabel('Daily Return (%)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

plt.suptitle('Daily Return Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/return_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.5 Outlier Detection

In [ ]:
print("Outlier Analysis (returns beyond ±3 standard deviations)\n")
print(f"{'Ticker':<8} {'Total Days':<12} {'Outliers':<10} {'% Outliers':<12} {'Max Return':<14} {'Min Return'}")
print('-' * 70)

for ticker in TICKERS:
    returns = dfs[ticker]['Daily_Return'].dropna()
    mean, std = returns.mean(), returns.std()
    outliers = returns[(returns > mean + 3*std) | (returns < mean - 3*std)]
    print(f"{ticker:<8} {len(returns):<12} {len(outliers):<10} {len(outliers)/len(returns)*100:<12.2f} {returns.max():<14.2f} {returns.min():.2f}")

In [ ]:
# Top 5 extreme return days for TSLA
tsla_returns = dfs['TSLA']['Daily_Return'].dropna()
print("TSLA - Top 5 Highest Return Days:")
print(tsla_returns.nlargest(5).to_frame())
print("\nTSLA - Top 5 Lowest Return Days:")
print(tsla_returns.nsmallest(5).to_frame())

### 4.6 Correlation Heatmap

In [ ]:
combined_returns = pd.DataFrame({
    ticker: dfs[ticker]['Daily_Return'] for ticker in TICKERS
}).dropna()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(combined_returns.corr(), annot=True, fmt='.3f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Daily Returns Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Seasonality and Trend Analysis

In [ ]:
# Seasonal decomposition for TSLA
tsla_monthly = dfs['TSLA']['Adj Close'].resample('ME').last()
decomposition = seasonal_decompose(tsla_monthly, model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10))
components = [('Observed', decomposition.observed),
              ('Trend', decomposition.trend),
              ('Seasonal', decomposition.seasonal),
              ('Residual', decomposition.resid)]

for ax, (label, data) in zip(axes, components):
    ax.plot(data.index, data, color='#E31937', linewidth=1.2)
    ax.set_ylabel(label, fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('TSLA - Seasonal Decomposition (Monthly)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/seasonal_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.1 Augmented Dickey-Fuller (ADF) Stationarity Test

In [ ]:
def adf_test(series, name):
    result = adfuller(series.dropna(), autolag='AIC')
    stationary = result[1] < 0.05
    print(f"  ADF Statistic : {result[0]:.4f}")
    print(f"  p-value       : {result[1]:.4f}")
    print(f"  Critical (5%) : {result[4]['5%']:.4f}")
    print(f"  Result        : {'STATIONARY ✓' if stationary else 'NON-STATIONARY ✗'}")
    return stationary

for ticker in TICKERS:
    print(f"\n{'='*50}")
    print(f"  {ticker} - Closing Price")
    print('='*50)
    adf_test(dfs[ticker]['Adj Close'], f'{ticker} Close')

    print(f"\n  {ticker} - Daily Returns")
    print('-'*50)
    adf_test(dfs[ticker]['Daily_Return'], f'{ticker} Returns')

**Interpretation:**
- Closing prices are typically **non-stationary** (p > 0.05) — they exhibit trends and are not mean-reverting. This means ARIMA will require differencing (d ≥ 1).
- Daily returns are typically **stationary** (p < 0.05) — they fluctuate around a constant mean, making them suitable for direct modeling.

## 6. Risk Metrics

In [ ]:
RISK_FREE_RATE_DAILY = 0.05 / 252  # 5% annual risk-free rate
CONFIDENCE_LEVEL = 0.95
TRADING_DAYS = 252

print(f"{'Metric':<30} {'TSLA':>12} {'BND':>12} {'SPY':>12}")
print('=' * 70)

metrics = {}
for ticker in TICKERS:
    returns = dfs[ticker]['Daily_Return'].dropna() / 100

    # Value at Risk (Historical)
    var_95 = np.percentile(returns, (1 - CONFIDENCE_LEVEL) * 100)

    # Sharpe Ratio (annualized)
    excess_returns = returns - RISK_FREE_RATE_DAILY
    sharpe = (excess_returns.mean() / excess_returns.std()) * np.sqrt(TRADING_DAYS)

    # Annualized Volatility
    ann_vol = returns.std() * np.sqrt(TRADING_DAYS)

    # Annualized Return
    ann_return = returns.mean() * TRADING_DAYS

    metrics[ticker] = {
        'Ann. Return': ann_return,
        'Ann. Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'VaR (95%)': var_95
    }

metrics_df = pd.DataFrame(metrics)
for metric in metrics_df.index:
    row = metrics_df.loc[metric]
    print(f"{metric:<30} {row['TSLA']:>12.4f} {row['BND']:>12.4f} {row['SPY']:>12.4f}")

In [ ]:
# VaR Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, ticker in zip(axes, TICKERS):
    returns = dfs[ticker]['Daily_Return'].dropna()
    var_95 = np.percentile(returns / 100, 0.05) * 100

    ax.hist(returns, bins=80, color=colors[ticker], edgecolor='white', alpha=0.75)
    ax.axvline(var_95, color='red', linestyle='--', linewidth=2,
               label=f'VaR 95%: {var_95:.2f}%')
    ax.set_title(f'{ticker} - Value at Risk', fontsize=12)
    ax.set_xlabel('Daily Return (%)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

plt.suptitle('Historical Value at Risk (95% Confidence)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/var_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Insights Summary

In [ ]:
print("="*65)
print("  KEY INSIGHTS SUMMARY")
print("="*65)

for ticker in TICKERS:
    returns = dfs[ticker]['Daily_Return'].dropna()
    price = dfs[ticker]['Adj Close']
    total_return = (price.iloc[-1] / price.iloc[0] - 1) * 100
    print(f"\n{ticker}:")
    print(f"  Total Return (period)  : {total_return:.1f}%")
    print(f"  Avg Daily Return       : {returns.mean():.4f}%")
    print(f"  Daily Volatility (Std) : {returns.std():.4f}%")
    print(f"  Sharpe Ratio           : {metrics[ticker]['Sharpe Ratio']:.4f}")
    print(f"  VaR 95% (daily)        : {metrics[ticker]['VaR (95%)']*100:.4f}%")

print("\n" + "="*65)
print("  TSLA: High growth, high volatility — dominant total return")
print("  BND : Stable, low volatility — capital preservation")
print("  SPY : Balanced risk/return — broad market diversification")
print("="*65)